# Cuál es el problema?

Abandono de clientes (Churn)

Trabajamos con una empresa de telecomunicaciones que quiere ***predecir cuales clientes van a abandonar el servicio*** (churn) para poder tomar acciones preventivas.

Por qué es importante?

* Conseguir un cliente nuevo representa entre 5x y 25x más que retener uno existente.
* Si podemos predecir quien se va, podemos ofrecerle un descuento o llamarlo antes de que cancele.



In [7]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split # Dividir datos en entrenamiento y prueba
from sklearn.compose import ColumnTransformer # Para transformar columnas específicas
from sklearn.preprocessing import OneHotEncoder # Para escalar y codificar variables
from sklearn.linear_model import LogisticRegression # Modelo de regresión logística

# Metricas de evaluación para la clasificación

from sklearn.metrics import (
    confusion_matrix, # Matriz de confusión
    classification_report, # reporte con precisión, recall y f1-score
    accuracy_score, # Porcentaje de predicciones correctas
)

import matplotlib.pyplot as plt
print("Librerias cargadas correctamente")

Librerias cargadas correctamente


In [6]:
%pip install pandas numpy scikit-learn matplotlib

  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached pyparsing-3.3.2-py3-none-any.whl.metadata (5.8 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.7 MB 4.7 MB/s eta 0:00:02
   ---------- ----------------------------- 2.6/9.7 MB 8.5 MB/s eta 0:00:01
   ------------------ --------------------- 4.5/9.7 MB 7.8 MB/s eta 0:00:01
   ----------------------------- ---------- 7.1/9.7 MB 9.0 MB/s eta 0:00:01
   -------------------------------------- - 9.4/9.7 MB 9.6 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 9.3 MB/s  0:00:01
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   ----- ---------------------------------- 1.8/12.3 MB 10.4 MB/s eta 0:00:02


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
#Cargar el archivo CSV en un DataFrame (tabla de datos  )
df = pd.read_csv("abandono_clientes.csv")
# Mostrar las primeras filas del DataFrame para verificar que se cargó correctamente
print("primeras 5 filas del DataSet:")
df.head()

primeras 5 filas del DataSet:


,antiguedad_meses,factura_mensual,llamadas_soporte,contrato,satisfaccion,abandono
0,52,61.18,0,Dos_anios,3,0
1,15,54.89,1,Mensual,1,1
2,72,112.95,3,Anual,2,0
3,61,103.06,0,Mensual,1,1
4,21,116.50,2,Mensual,1,1


In [10]:
# Informacion general del dataset
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
print()

# Tipo de dato de cada columna
print("Tipos de datos:")
for col, dtype in df.dtypes.items():
    print(f"  {col:20s} -> {dtype}")

print()

# Verificar valores faltantes
nulos = df.isna().sum()
if nulos.sum() == 0:
    print("No hay valores faltantes en el dataset.")
else:
    print(f"ATENCION: hay {nulos.sum()} valores nulos.")
    print(nulos[nulos > 0])

Filas: 800, Columnas: 6

Tipos de datos:
  antiguedad_meses     -> int64
  factura_mensual      -> float64
  llamadas_soporte     -> int64
  contrato             -> str
  satisfaccion         -> int64
  abandono             -> int64

No hay valores faltantes en el dataset.


In [11]:
conteo = df["abandono"].value_counts()
porcentaje = df["abandono"].value_counts(normalize=True) * 100

print("Distribucion de la variable objetivo (abandono):")
print(f"  Permanece (0): {conteo[0]} clientes ({porcentaje[0]:.1f}%)")
print(f"  Abandona  (1): {conteo[1]} clientes ({porcentaje[1]:.1f}%)")
print()
print("Las clases estan moderadamente desbalanceadas.")
print("Hay mas clientes que permanecen que clientes que abandonan.")
print("Esto es lo tipico en la realidad: la mayoria de clientes NO se van.")

Distribucion de la variable objetivo (abandono):
  Permanece (0): 517 clientes (64.6%)
  Abandona  (1): 283 clientes (35.4%)

Las clases estan moderadamente desbalanceadas.
Hay mas clientes que permanecen que clientes que abandonan.
Esto es lo tipico en la realidad: la mayoria de clientes NO se van.


# Analisis exporatorio: Como se comportan las variables segun abandono?

## Antes de entrana el modelo, vemos si las variables tiene relación con el abandono. Esto nos ayuda a entender que factores podrian influir en la decisión del cliente

In [14]:
#Comporar el promedio de cada variable 

comparacion = df.groupby("abandono")[["antiguedad_meses","factura_mensual",
                                      "llamadas_soporte","satisfaccion"]].mean().round(2)
comparacion.index=["Permanece (0)","Abandona (1)"]
print("Promedio de variables numerica segun abandono:")
comparacion

Promedio de variables numerica segun abandono:


,antiguedad_meses,factura_mensual,llamadas_soporte,satisfaccion
Permanece (0),39.43,67.02,1.88,3.14
Abandona (1),28.41,77.27,2.28,2.67


In [16]:
# Analisi del tipo contrato vs el abandono
tabla_contrato = pd.crosstab(
    df["contrato"], 
    df["abandono"], 
    normalize="index"
    ) * 100
tabla_contrato.columns = ["Permanece (%)", "Abandona (%)"]
print("porcentaje de abandono pro tipo de contrato:")
comparacion 

porcentaje de abandono pro tipo de contrato:


,antiguedad_meses,factura_mensual,llamadas_soporte,satisfaccion
Permanece (0),39.43,67.02,1.88,3.14
Abandona (1),28.41,77.27,2.28,2.67


# Preapara los datos para el modelo.

1. Separar variables predictoras (x) de la variable objetivo (y)
2. Entrenar los datoas con el (80%) del dataset y prueba con el (20%) del dataset



In [19]:
# Paso 1 separar X(vriables predictoas) e y (variabl objetivo)
X= df[["antiguedad_meses","factura_mensual","llamadas_soporte",
        "satisfaccion","contrato"]]
y=df["abandono"]

# Paso 2 Dividir el conjunto de entrenamiento (80%) y prueba (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
     test_size=0.2, # 20% para prueba
     random_state=42, # 80% para reproducibilidad
        stratify=y
)

print(f"Datos de entrenamiento: {X_train.shape[0]} Clientes")
print(f"Datos de prueba: {X_test.shape[0]} Clientes")
print("\n")
print(f"Proporcion de abandono en entrenamiento: {y_train.mean()*100:.1f}%")
print(f"Proporcion de abandono en prueba: {y_test.mean()*100:.1f}%")
print()
print("La proporciones son similares gracias al parametro 'stratify=y' en train_test_split")

Datos de entrenamiento: 640 Clientes
Datos de prueba: 160 Clientes


Proporcion de abandono en entrenamiento: 35.3%
Proporcion de abandono en prueba: 35.6%

La proporciones son similares gracias al parametro 'stratify=y' en train_test_split


# Construir  el pipeline y entrenar el modelo

1. Calula una puntuación lineal
2.Pasa por una puntación que la convierte en probabilidad entre 0 y 1
3. SI la probabilidad es mayor a 0.5  predice la clase 1 (abandona): si no, predice 0 (permanece)

In [23]:
from sklearn.pipeline import Pipeline

numeric_features = ["antiguedad_meses", "factura_mensual",
                    "llamadas_soporte", "satisfaccion"]
categorical_features = ["contrato"]


preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)


pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000)),
])

# Entrenar el modelo con los datos de entrenamiento
pipe.fit(X_train, y_train)

print("Modelo entranado correctamente.")


Modelo entranado correctamente.


# Hacer predicciones
Usamos el modulo entranado para predecir el abandono en los datos de pruebas 

In [25]:
# Generar prediciones con los datos de prueba
y_pred = pipe.predict(X_test)

# Mostrar las primeras 10 predicciones
comparacion_pred = pd.DataFrame({
    "Real": y_test.values[:10],
    "Prediccion": y_pred[:10],
    "Correcto": ["Si" if r == p else "No" for r, p in zip(y_test.values[:10], y_pred[:10])]
})

print("Primeras 10 predicciones vs realidad:")
print()
comparacion_pred

Primeras 10 predicciones vs realidad:



,Real,Prediccion,Correcto
0,1,1,Si
1,0,0,Si
2,0,0,Si
3,0,0,Si
4,0,0,Si
5,0,0,Si
6,0,0,Si
7,0,0,Si
8,0,0,Si
9,0,0,Si


# Evaluación del modulo:  exactitud/presución (Acurracy)

Cuando hablamos del porcentaje de presición nos referimos a las predicciones que fueron correctas

Accurracy= predicciones Correctas /Total de predicciones.

Ejemplo: Si el 95% de clientes **NO** abandona, un modelo que siempre diga "No abandona" tendria un 95% de presicion, pero jamas detectaria a un cliente en riegos. seria inutil.





In [26]:
acc=accuracy_score(y_test, y_pred)
print(f"Accuracy (exactitud): {acc:.4f} ({acc*100:.2f}%%)")
print()
print(f"Esto significa que el modulo acerto {acc*100:.2f}% predicciones.")

Accuracy (exactitud): 0.7625 (76.25%%)

Esto significa que el modulo acerto 76.25% predicciones.
